# Análisis Exploratorio y Selección de Instancias: PACE 2017 Track B

## 1. Contexto del Estudio
Este notebook tiene como objetivo realizar la selección de instancias de prueba para la evaluación del **Algoritmo Evolutivo (AE)** desarrollado para el problema de *Minimum Fill-in* (Cordalización de Grafos).

Para asegurar la validez científica y la comparabilidad de los resultados, se utilizan grafos provenientes del **PACE 2017 Challenge (Track B)**, una competencia internacional dedicada específicamente a este problema. A diferencia de los grafos generados aleatoriamente, estas instancias representan estructuras reales y desafíos topológicos estandarizados por la comunidad científica.

## 2. Objetivos del Notebook
1.  **Escaneo de Metadatos:** Leer las propiedades topológicas básicas (Número de Nodos $|V|$, Número de Aristas $|E|$ y Densidad $D$) de todo el conjunto de datos sin cargar la estructura completa en memoria.
2.  **Filtrado de Viabilidad:** Descartar instancias triviales (demasiado pequeñas) o computacionalmente intratables para una implementación en Python puro (instancias masivas de >1,000 nodos).
3.  **Análisis de Distribución:** Visualizar la dispersión de los grafos en función de su tamaño y densidad para identificar clústeres representativos.
4.  **Selección Estratégica:** Seleccionar un subconjunto de **9 instancias** que cubran distintos grados de dificultad, siguiendo las directrices de diseño experimental de Eiben & Smith (2015):
    * **3 Instancias Pequeñas (Small):** Validación funcional y convergencia rápida.
    * **3 Instancias Medianas (Medium):** Análisis de robustez paramétrica.
    * **3 Instancias Grandes (Large):** Prueba de estrés y escalabilidad.

## 3. Criterios de Selección
Dado que el problema de *Minimum Fill-in* es NP-Completo, el tiempo de cómputo crece exponencialmente con el tamaño del grafo. Para esta tesis de maestría, se definen los siguientes rangos de interés:

| Categoría | Rango de Nodos ($N$) | Propósito Experimental |
| :--- | :--- | :--- |
| **Small** | $20 \le N \le 60$ | Depuración, Sanity Check y validación de optimalidad. |
| **Medium** | $61 \le N \le 150$ | **Núcleo del estudio.** Balance entre complejidad y tiempo de ejecución. |
| **Large** | $151 \le N \le 400$ | Límite superior de la implementación actual. Evaluación de crecimiento del AES (Average Evaluations to Solution). |

## 4. Origen de los Datos
* **Fuente:** [PACE 2017 Track B Repository](https://pacechallenge.org/2017/minimum-fill-in/)
* **Formato:** DIMACS (`.graph`)
* **Métrica Objetivo:** *Minimum Fill-in* (Número de aristas añadidas para volver al grafo cordal).

---
*Nota: La selección busca maximizar la diversidad topológica dentro de cada categoría (ej. seleccionar grafos dispersos y densos dentro del mismo rango de tamaño).*

In [31]:
import os
import pandas as pd
import networkx as nx
import numpy as np
from tqdm.auto import tqdm

# Configuración visual de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

PACE_DIR = "../data/raw/PACE2017B"


def load_pace_graph(filepath):
    """
    Carga grafo desde archivo sin header, ignorando comentarios.
    Convierte etiquetas a enteros consecutivos.
    """
    G = nx.Graph()
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('#'): continue
            parts = line.split()
            u, v = parts[0], parts[1]
            G.add_edge(u, v)

    # Convertir nodos a int 0..N-1 para velocidad
    G = nx.convert_node_labels_to_integers(G)
    return G

def run_greedy_min_degree_heuristic(G_input):
    """
    Implementación explícita de la heurística Minimum Degree.
    1. Busca el nodo con menor grado.
    2. Conecta a sus vecinos entre sí (añade fill-in).
    3. Elimina el nodo y repite.
    Retorna: Cantidad de aristas añadidas (Fill-in).
    """
    # Trabajamos sobre una copia para no romper el original
    G = G_input.copy()
    fill_in_count = 0

    # Mientras queden nodos...
    # (Nota: Esta implementación es O(N^3) en el peor caso,
    # pero para N<500 es cuestión de milisegundos).

    # Optimizamos un poco usando la estructura de datos interna
    nodes_left = list(G.nodes())

    while len(G) > 0:
        # 1. Encontrar nodo de grado mínimo en el grafo actual
        # (Usamos una lambda simple, eficiente para grafos dispersos)
        min_node = min(G.nodes(), key=lambda n: G.degree[n])

        # 2. Obtener vecinos
        neighbors = list(G.neighbors(min_node))

        # 3. Conectar vecinos entre sí (formar clique)
        # Esto es lo que genera el "Fill-in"
        for i in range(len(neighbors)):
            u = neighbors[i]
            for j in range(i + 1, len(neighbors)):
                v = neighbors[j]
                if not G.has_edge(u, v):
                    G.add_edge(u, v)
                    fill_in_count += 1

        # 4. Eliminar nodo del grafo
        G.remove_node(min_node)

    return fill_in_count

def process_graph_metrics(filepath):
    """Procesa un solo archivo y calcula métricas + heurística."""
    filename = os.path.basename(filepath)
    G = load_pace_graph(filepath)

    if G is None or G.number_of_nodes() == 0:
        return None

    n = G.number_of_nodes()
    m = G.number_of_edges()
    density = (2 * m) / (n * (n - 1)) if n > 1 else 0

    # Calcular Heurística solo si el grafo es viable (ej. < 1000 nodos)
    # para no bloquear la máquina si hay un grafo monstruoso.
    if n <= 1000:
        heuristic_val = run_greedy_min_degree_heuristic(G)
    else:
        heuristic_val = -2 # Código para "Demasiado grande para calcular rápido"

    return {
        "Filename": filename,
        "Nodes": n,
        "Edges": m,
        "Density": density,
        "MinDegree_FillIn": heuristic_val, # Nombre más exacto
        "Format": "PACE String Labels",
        "Path": filepath
    }

# --- EJECUCIÓN PRINCIPAL ---

print(f"📂 Escaneando directorio: {PACE_DIR}")
files = [f for f in os.listdir(PACE_DIR) if f.endswith('.gr') or f.endswith('.graph')]
print(f"📊 Procesando {len(files)} archivos...")

data = []
# Barra de carga bonita
for f in tqdm(files):
    path = os.path.join(PACE_DIR, f)
    row = process_graph_metrics(path)
    if row:
        data.append(row)

df = pd.DataFrame(data)

# Ordenar y mostrar
df = df.sort_values(by="Nodes").reset_index(drop=True)
print("✅ Procesamiento completado.")
display(df.head(10))

📂 Escaneando directorio: ../data/raw/PACE2017B
📊 Procesando 100 archivos...


  0%|          | 0/100 [00:00<?, ?it/s]

✅ Procesamiento completado.


,Filename,Nodes,Edges,Density,MinDegree_FillIn,Format,Path
0,43.graph,27,69,0.1966,0,PACE String Labels,../data/raw/PACE2017B\43.graph
1,9.graph,27,254,0.7236,9,PACE String Labels,../data/raw/PACE2017B\9.graph
2,74.graph,30,307,0.7057,0,PACE String Labels,../data/raw/PACE2017B\74.graph
3,50.graph,30,349,0.8023,4,PACE String Labels,../data/raw/PACE2017B\50.graph
4,75.graph,40,539,0.6910,3,PACE String Labels,../data/raw/PACE2017B\75.graph
5,51.graph,40,587,0.7526,5,PACE String Labels,../data/raw/PACE2017B\51.graph
6,20.graph,50,59,0.0482,2,PACE String Labels,../data/raw/PACE2017B\20.graph
7,42.graph,66,1460,0.6807,12,PACE String Labels,../data/raw/PACE2017B\42.graph
8,76.graph,72,1121,0.4386,25,PACE String Labels,../data/raw/PACE2017B\76.graph
9,24.graph,75,158,0.0569,65,PACE String Labels,../data/raw/PACE2017B\24.graph


In [25]:
def get_category(n):
    if n <= 60: return 'Small'
    if n <= 150: return 'Medium'
    return 'Large'

df_viable['Category'] = df_viable['Nodes'].apply(get_category)

print(f"Calculando heurística AMD para {len(df_viable)} grafos seleccionados...")
# Usamos tqdm para ver barra de progreso
tqdm.pandas()
df_viable['AMD_FillIn'] = df_viable['Path'].progress_apply(calculate_amd_fillin)

# Calculamos un "Ratio de Dificultad" (Fill-in por cada Arista original)
# Un ratio alto sugiere un grafo complejo estructuralmente
df_viable['Difficulty_Ratio'] = df_viable['AMD_FillIn'] / df_viable['Edges']

print("✅ Cálculo completado.")

Calculando heurística AMD para 79 grafos seleccionados...


KeyError: 'Path'